# FHIR Repository — Part 7: RESTful Events and Aggregates

`03-laboratory-order-from-csv.ipynb` through `06-eu-laboratory-report-fhir-document.ipynb`
all built **FHIR Messaging** `Bundle`s (`Bundle.type = "message"`/`"document"`) — the
format this repo's own NW Regional Integration Engine (RIE) speaks, and posted them at
`$process-message`. Separately, `01-fhir-search-basics.ipynb` and
`02-work-orders-worked-example.ipynb` (copied in from a sibling project, `nw-gmsa/julius`
— they talk about "this app"'s own `fhir_client.py`/`/order/new`, which don't exist in
*this* repo, so treat them as read-only reference material rather than this series'
house style) already toured **FHIR RESTful** `GET` against this same repository —
including, in `02`'s section 5, building the JSON Patch/`If-Match` body for a `Task`
status update, deliberately without ever sending it.

This notebook is the one that actually sends a write. It picks up `03`-`06`'s own
running example — `ctdna9737383222.txt`, NHS number `9737383222`, "Rob Leeds" — and
shows the other side of the same coin: not building a message for the RIE to process,
but calling the FHIR Repository directly, the way [NHS England's Genomic Order
Management Service
(GOMS)](https://digital.nhs.uk/developer/api-catalogue/genomic-order-management-service-fhir)
expects a consumer to.

## 1. RESTful vs. Messaging, and where this notebook fits

A FHIR Repository — an off-the-shelf product such as AWS HealthLake or, here, an
InterSystems HealthConnect Clinical Data Repository (CDR) — supports the full FHIR
RESTful interaction set: `read`, `vread`, `create`, `update`, `patch`, `delete`,
`history`, `search`, plus system-level `batch`/`transaction`. That's NHS England's own
preferred way of talking to GOMS-shaped services. It's also, deliberately, *not* how NW
GM-SA expects external customers to talk to **its own** CDR:

- **Internally**, the RIE is free to be as chatty as it needs to be — verifying a
  patient against PDS, updating the local mini-MPI, working out which of several
  candidate `ServiceRequest`s a result belongs to — all of it real RESTful traffic
  against the CDR, just never seen outside the RIE process. `03`-`06`'s `$process-message`
  calls are the *messaging* face of that same internal traffic.
- **Externally**, NW GM-SA does not expose RESTful `POST`/`PUT`/`PATCH` at all — only
  read-only `GET`. The business logic behind *how* an aggregate got to its current
  state (which PDS check ran, which local identifier got attached and why) stays
  internal; an external consumer only ever sees the result.

![notebook 7 diagram 1](https://mermaid.ink/svg/Zmxvd2NoYXJ0IExSCiAgICBzdWJncmFwaCBFWFRbRXh0ZXJuYWwgR09NUy1zdHlsZSBjb25zdW1lcl0KICAgICAgICBHW0dFVCBvbmx5PGJyLz5yZWFkLW9ubHkgUkVTVGZ1bF0KICAgIGVuZAogICAgc3ViZ3JhcGggUklFW05XIFJlZ2lvbmFsIEludGVncmF0aW9uIEVuZ2luZV0KICAgICAgICBNW1YyIC8gRkhJUiBNZXNzYWdpbmc8YnIvPkxBQi0xIC4gTEFCLTQgLiBMQUItNSAuIExBQi0zPGJyLz5oaWRlcyBQT1NUL1BVVC9QQVRDSCwgUERTIGNoZWNrcyw8YnIvPmxvY2FsIE1QSSBtYXRjaGluZ10KICAgIGVuZAogICAgc3ViZ3JhcGggQ0RSW0ZISVIgUmVwb3NpdG9yeTxici8+SW50ZXJTeXN0ZW1zIEhlYWx0aENvbm5lY3QgQ0RSXQogICAgICAgIEFbKFBhdGllbnQsIFNlcnZpY2VSZXF1ZXN0LDxici8+U3BlY2ltZW4sIE9ic2VydmF0aW9uLDxici8+RGlhZ25vc3RpY1JlcG9ydCldCiAgICBlbmQKICAgIEcgLS0+IEEKICAgIE0gLS0+fGludGVybmFsIFJFU1RmdWw8YnIvPlBPU1QgLyBQVVQgLyBQQVRDSCAvIHRyYW5zYWN0aW9ufCBBCg==)

<!--
```mermaid
flowchart LR
    subgraph EXT[External GOMS-style consumer]
        G[GET only<br/>read-only RESTful]
    end
    subgraph RIE[NW Regional Integration Engine]
        M[V2 / FHIR Messaging<br/>LAB-1 . LAB-4 . LAB-5 . LAB-3<br/>hides POST/PUT/PATCH, PDS checks,<br/>local MPI matching]
    end
    subgraph CDR[FHIR Repository<br/>InterSystems HealthConnect CDR]
        A[(Patient, ServiceRequest,<br/>Specimen, Observation,<br/>DiagnosticReport)]
    end
    G --> A
    M -->|internal RESTful<br/>POST / PUT / PATCH / transaction| A
```
-->

### Framing this as Domain-Driven Design

Each incoming message — `LAB-1`, `LAB-4`, `LAB-5`, `LAB-3` — is a domain **event**. Each
FHIR resource it touches — `Patient`, `ServiceRequest`, `Specimen`, `Observation`,
`DiagnosticReport` — is an **aggregate** that the event either creates, mutates a small
part of, or merely references without changing at all:

![notebook 7 diagram 2](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgICBzdWJncmFwaCBMMVtMQUItMSBMYWJvcmF0b3J5IE9yZGVyXQogICAgICAgIFAxW1BhdGllbnQ6IHNlYXJjaCBieSBpZGVudGlmaWVyLDxici8+UEFUQ0ggdG8gYWRkIGFuIGlkZW50aWZpZXJdCiAgICBlbmQKICAgIHN1YmdyYXBoIEw0W0xBQi00IFdvcmsgT3JkZXIgLSBvcHRpb25hbF0KICAgICAgICBTMVtTZXJ2aWNlUmVxdWVzdDogc3RhdHVzIC0+IGFjdGl2ZS9yZXZpc2VkPGJyLz5ubyBsaXZlIGNhbGwgaW4gdGhpcyBub3RlYm9va10KICAgIGVuZAogICAgc3ViZ3JhcGggTDVbTEFCLTUgVGVzdCBSZXN1bHRzXQogICAgICAgIE8xW09ic2VydmF0aW9uIHg0OiB2YXJpYW50IGNhbGxzPGJyLz5QT1NUIGFzIG9uZSB0cmFuc2FjdGlvbiBCdW5kbGVdCiAgICBlbmQKICAgIHN1YmdyYXBoIEwzW0xBQi0zIExhYm9yYXRvcnkgUmVwb3J0XQogICAgICAgIFMyW1NlcnZpY2VSZXF1ZXN0OiBzdGF0dXMgLT4gY29tcGxldGVkPGJyLz5QQVRDSCB3aXRoIElmLU1hdGNoXQogICAgZW5kCiAgICBMMSAtLT4gTDQgLS0+IEw1IC0tPiBMMwo=)

<!--
```mermaid
flowchart TB
    subgraph L1[LAB-1 Laboratory Order]
        P1[Patient: search by identifier,<br/>PATCH to add an identifier]
    end
    subgraph L4[LAB-4 Work Order - optional]
        S1[ServiceRequest: status -> active/revised<br/>no live call in this notebook]
    end
    subgraph L5[LAB-5 Test Results]
        O1[Observation x4: variant calls<br/>POST as one transaction Bundle]
    end
    subgraph L3[LAB-3 Laboratory Report]
        S2[ServiceRequest: status -> completed<br/>PATCH with If-Match]
    end
    L1 --> L4 --> L5 --> L3
```
-->

### Why this notebook writes into real, already-processed data

`ctdna9737383222.txt` is the same fixture `04`/`05`/`06` used. This repository's own
production pipeline had already processed it before this notebook was written: real
`Patient`, `ServiceRequest`, `Specimen`, `Observation` and `DiagnosticReport` resources
already exist for it (`Patient/203499`, `ServiceRequest/205266`, `Specimen/206851`,
`Observation/205268`, `DiagnosticReport/205267`). Rather than inventing a fresh,
isolated test patient, every write below targets that real data — the point being to
demonstrate the RESTful event pattern against a genuine aggregate graph, using search
to locate the right resource before deciding create vs. update vs. reference-only,
exactly as a real GOMS-style consumer would have to.

**Every write is additive and safe to re-run**: an identifier `add` on `Patient` is
skipped if already present; the `Observation`s in section 6 use a conditional
`ifNoneExist` create so a repeat run doesn't duplicate them; the `status` `PATCH` in
section 7 sets a value that's already true. Nothing here deletes or overwrites existing
data.

**Each of sections 4, 5, 6 and 7 is self-contained.** `LAB-1`/`LAB-4`/`LAB-5`/`LAB-3` are
genuinely separate events that in real life arrive hours or days apart, from different
systems, quite possibly in a different process or a different day's notebook run — so
each section's first cell re-discovers everything it needs by searching the server cold
(`ServiceRequest?identifier=...`, `Patient?identifier=...`), rather than reusing a Python
variable a previous section's cell happened to leave lying around. Only section 6 reads
a reference (the `ServiceRequest`'s own `id`) that section 4 also independently looked
up — and it re-looks it up itself rather than trusting section 4's copy.

## 2. Connect, and see what this server actually supports

`FHIR_BASE_URL`/`FHIR_USER`/`FHIR_PASSWORD` are already in `.env` — the same host
`01`/`02` point at, a different path on the same box as `03`-`06`'s `FHIR_SERVER`
(that one fronts `$process-message`; this one is the CDR's plain FHIR RESTful
endpoint). HTTP Basic auth, same as `01`/`02`.

The helper functions below are this notebook's equivalent of `05`/`06`'s
`validate_resource()` — small wrappers reused in every section from here on, rather
than repeating `requests` boilerplate five times.

In [1]:
import json
import os
import subprocess
import tempfile
from pathlib import Path
from uuid import uuid4

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["FHIR_BASE_URL"]
AUTH = (os.environ["FHIR_USER"], os.environ["FHIR_PASSWORD"])
VERIFY_SSL = os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true"

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
GENOMICS_VARIANT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"
GENOMICS_REPORTING_IG = "hl7.fhir.uv.genomics-reporting#3.0.0"

# ctdna9737383222.txt's own identity, same fixture as 04/05/06 - Rob Leeds, NHS 9737383222.
NHS_NUMBER = "9737383222"
SERVICE_REQUEST_PLACER_NUMBER = "1234-RR8"  # ORC-2 in the v2 fixture


def fhir_get(path, params=None):
    resp = requests.get(
        f"{BASE_URL}/{path}", params=params, auth=AUTH,
        headers={"Accept": "application/fhir+json"}, verify=VERIFY_SSL, timeout=30,
    )
    resp.raise_for_status()
    return resp


def fhir_search(resource_type, params):
    bundle = fhir_get(resource_type, params).json()
    return [entry["resource"] for entry in bundle.get("entry", []) if "resource" in entry]


def fhir_patch(resource_type, resource_id, patch_ops, etag):
    return requests.patch(
        f"{BASE_URL}/{resource_type}/{resource_id}",
        data=json.dumps(patch_ops), auth=AUTH,
        headers={"Content-Type": "application/json-patch+json", "If-Match": etag},
        verify=VERIFY_SSL, timeout=30,
    )


def fhir_transaction(bundle):
    return requests.post(
        BASE_URL, data=json.dumps(bundle), auth=AUTH,
        headers={"Content-Type": "application/fhir+json"}, verify=VERIFY_SSL, timeout=30,
    )


def validate_resource(resource, igs, profiles):
    # Same dev-loop check 03-06 use before ever sending something onward.
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = Path(tmp.name)
    output = Path(str(tmp_path) + "-OperationOutcome.json")
    args = ["java", "-jar", "validator_cli.jar", str(tmp_path), "-version", "4.0.1", "-tx", "n/a"]
    for ig in igs:
        args += ["-ig", ig]
    for profile in profiles:
        args += ["-profile", profile]
    args += ["-output", str(output), "-output-style", "json"]
    subprocess.run(args, capture_output=True)
    with open(output) as f:
        outcome = json.load(f)
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(lambda item: item["text"])
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    # Same noise this repo's validator always produces without a live terminology
    # server - filtered the same way 03-06 already do, not specific to this notebook.
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    df = df[~df["details"].str.contains("no terminology service")]
    return df.sort_values(by=["severity"])


print(f"BASE_URL = {BASE_URL}")

BASE_URL = https://192.168.1.62/healthconnect/cdr/fhir/r4


### What does this deployment actually let us do?

Don't assume - `GET /metadata` returns a `CapabilityStatement` listing exactly which
interactions each resource type supports. This is what a real integration would check
before assuming `patch` or `transaction` are even available.

In [2]:
capability = fhir_get("metadata").json()
resources_of_interest = {"Patient", "Encounter", "ServiceRequest", "Specimen", "Observation", "DiagnosticReport"}
rest = capability["rest"][0]

rows = []
for resource in rest["resource"]:
    if resource["type"] in resources_of_interest:
        rows.append({
            "resourceType": resource["type"],
            "interactions": ", ".join(i["code"] for i in resource.get("interaction", [])),
            "conditionalCreate": resource.get("conditionalCreate", False),
            "conditionalUpdate": resource.get("conditionalUpdate", False),
        })
system_interactions = ", ".join(i["code"] for i in rest.get("interaction", []))
print(f"System-level interactions: {system_interactions}")
pd.DataFrame(rows)

System-level interactions: transaction, batch, search-system


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,resourceType,interactions,conditionalCreate,conditionalUpdate
0,DiagnosticReport,"read, vread, create, update, patch, delete, hi...",True,True
1,Encounter,"read, vread, create, update, patch, delete, hi...",True,True
2,Observation,"read, vread, create, update, patch, delete, hi...",True,True
3,Patient,"read, vread, create, update, patch, delete, hi...",True,True
4,ServiceRequest,"read, vread, create, update, patch, delete, hi...",True,True
5,Specimen,"read, vread, create, update, patch, delete, hi...",True,True


`patch` and `create`/`update` are listed per resource type, `transaction`/`batch` show
up at the system level, and `conditionalCreate`/`conditionalUpdate` on `Observation`
confirm this deployment supports the `ifNoneExist` conditional-create pattern section 6
relies on below - checked, not assumed.

## 3. Look at what's already there

Read-only `GET`s only in this section — the same kind of call `01`/`02` already toured,
just walking the graph a real GLH system already built from this exact order: `Patient`
← `ServiceRequest` → `Specimen`, with `Observation`/`DiagnosticReport` referencing the
`ServiceRequest` via `basedOn`.

In [3]:
patients = fhir_search("Patient", {"identifier": NHS_NUMBER})
assert len(patients) == 1, f"expected exactly one Patient for NHS number {NHS_NUMBER}"
patient = patients[0]

service_requests = fhir_search("ServiceRequest", {"identifier": SERVICE_REQUEST_PLACER_NUMBER})
assert len(service_requests) == 1, f"expected exactly one ServiceRequest for placer number {SERVICE_REQUEST_PLACER_NUMBER}"
service_request = service_requests[0]

specimens = fhir_search("Specimen", {"subject": f"Patient/{patient['id']}"})
observations = fhir_search("Observation", {"subject": f"Patient/{patient['id']}"})
diagnostic_reports = fhir_search("DiagnosticReport", {"subject": f"Patient/{patient['id']}"})

name = (patient.get("name") or [{}])[0]
# Real registries carry data-quality noise - this Patient's own given-name array
# includes a literal "null" string, not a Python None, from whatever upstream
# system populated it. Worth filtering rather than printing verbatim.
given = [g for g in name.get("given", []) if g and g.lower() != "null"]
display_name = " ".join([*given, name.get("family", "")]).strip()

print(f"Patient/{patient['id']}  {display_name}  {len(patient.get('identifier', []))} identifier(s)")
print(f"ServiceRequest/{service_request['id']}  status={service_request['status']}  code={service_request['code']['coding'][0]['code']}")
for s in specimens:
    print(f"Specimen/{s['id']}  identifier={s['identifier'][0]['value']}  status={s['status']}")
for o in observations:
    print(f"Observation/{o['id']}  code={o['code']['coding'][0]['code']} ({o['code']['coding'][0].get('display', '')})")
for d in diagnostic_reports:
    print(f"DiagnosticReport/{d['id']}  status={d['status']}  code={d['code']['coding'][0]['code']}")

Patient/203499  Rob LEEDS  6 identifier(s)
ServiceRequest/205266  status=completed  code=M4.14
Specimen/206851  identifier=S26-2008  status=available
Observation/203503  code=81306-3 (Variables that apply to the overall study)
Observation/205268  code=81306-3 (Variables that apply to the overall study)
Observation/215599  code=69548-6 (Genetic variant assessment)
Observation/215600  code=69548-6 (Genetic variant assessment)
Observation/215601  code=69548-6 (Genetic variant assessment)
Observation/215602  code=69548-6 (Genetic variant assessment)
DiagnosticReport/203502  status=final  code=R125_Cardiology
DiagnosticReport/205267  status=final  code=ctDNA_M4


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised

This is the aggregate graph the rest of this notebook operates on. Nothing here has
been created for this notebook — it's the real output of this repo's own production
pipeline having processed `ctdna9737383222.txt` at some point before this notebook was
written.

## 4. `LAB-1` Laboratory Order — locate-then-update the `Patient` aggregate

`LAB-1` is the clinical requisition a clinician places. The first thing a real handler
does is work out whether the patient already exists — never blindly `POST` a new
`Patient` before checking. This cell makes no assumption that section 3 already ran:
it searches cold, by identifier, exactly as a handler receiving this event on its own
would have to.

In [4]:
# Cold lookup - assumes nothing about what's already in memory from section 3.
lab1_patients = fhir_search("Patient", {"identifier": NHS_NUMBER})
assert len(lab1_patients) == 1, f"expected exactly one Patient for NHS number {NHS_NUMBER}"
lab1_patient_id = lab1_patients[0]["id"]

print(f"Located Patient/{lab1_patient_id} - the aggregate already exists, so LAB-1's job here is")
print("to update it, not create a duplicate.")

Located Patient/203499 - the aggregate already exists, so LAB-1's job here is
to update it, not create a duplicate.


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### "Patient may get updated to add new identifiers"

A `LAB-1` can carry an identifier the Patient record doesn't have yet — e.g. a local
hospital MRN, or confirmation that PDS verification ran. This deployment's real
`Patient/203499` already carries five identifiers (an NHS number, two MRNs from
different assigning trusts, and a GLH-local patient identifier) picked up from earlier
real orders, and this particular fixture's own `PID-3` doesn't introduce anything new —
so rather than inventing a fake NHS number or MRN value that could be mistaken for
genuine patient data, this cell adds a clearly-scoped, harmless demo identifier under a
system that only this notebook uses.

The check-before-`PATCH` below is the idempotency a real event handler needs: `LAB-1`
redelivery (a message arriving twice, e.g. after a retry) must not append the same
identifier twice.

In [5]:
NOTEBOOK07_IDENTIFIER_SYSTEM = "https://fhir.nwgenomics.nhs.uk/Id/Notebook07DemoIdentifier"
NOTEBOOK07_IDENTIFIER_VALUE = "LAB1-EVENT-DEMO"

fresh_patient = fhir_get(f"Patient/{lab1_patient_id}")
patient_etag = fresh_patient.headers["ETag"]
patient_body = fresh_patient.json()

already_present = any(
    ident.get("system") == NOTEBOOK07_IDENTIFIER_SYSTEM and ident.get("value") == NOTEBOOK07_IDENTIFIER_VALUE
    for ident in patient_body.get("identifier", [])
)

if already_present:
    print(f"Patient/{lab1_patient_id} already carries this identifier - LAB-1 redelivery, correctly a no-op.")
else:
    patch_ops = [{
        "op": "add",
        "path": "/identifier/-",
        "value": {"system": NOTEBOOK07_IDENTIFIER_SYSTEM, "value": NOTEBOOK07_IDENTIFIER_VALUE},
    }]
    resp = fhir_patch("Patient", lab1_patient_id, patch_ops, patient_etag)
    print(f"PATCH Patient/{lab1_patient_id}  If-Match: {patient_etag}  -> {resp.status_code}")

    # Verify the write actually landed, rather than trusting the status code alone.
    verify = fhir_get(f"Patient/{lab1_patient_id}").json()
    added = [i for i in verify["identifier"] if i.get("system") == NOTEBOOK07_IDENTIFIER_SYSTEM]
    print(f"Confirmed via a fresh GET: {len(added)} matching identifier now present.")

/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Patient/203499 already carries this identifier - LAB-1 redelivery, correctly a no-op.


## 5. `LAB-4` Work Order — optional, and why

`LAB-4` is the LIMS's own instruction to a sequencer/bioinformatics pipeline: which
specimen, which tests, which analyser/pipeline configuration. It's internal to one
system (the LIMS talking to its own automation), not something every lab workflow
surfaces as a distinct externally-visible message — which is exactly why `05`'s own
notes call the order that actually triggers a `LAB-5` "more likely a `LAB-4` than a
`LAB-1`".

If a `LAB-4` did arrive as its own event here, it would touch the `ServiceRequest`
aggregate's `status` (`active` while work is in progress, possibly revised if the
work order itself changes) — no new resource, the same aggregate section 4 already
located. There's no live call in this section: unlike `LAB-1`/`LAB-5`/`LAB-3`, this
repository has no separate `LAB-4` fixture to source a realistic transition from, and
the real `ServiceRequest/205266` is already `completed` — moving it back to `active`
here just to demonstrate the mechanics would misrepresent this resource's genuine
history. `02`'s section 5 (`Task.status` transitions, also deliberately unsent) is the
closest existing worked example of the same `status`-lifecycle mechanics.

## 6. `LAB-5` Test Results — new aggregates via a transaction `Bundle`

A day or more after `LAB-4`, the sequencer/pipeline hands discrete variant calls back.
This section reuses `05`'s own VCF parsing and `build_observation()` unchanged (see
that notebook for the per-field mapping rationale) — same VCF, same function — the
only difference is *how* the results get to the server: as a live `POST` transaction
here, rather than wrapped in a message `Bundle` for `$process-message`.

Cold lookup first, same principle as section 4: this section doesn't reuse `service_request`
from section 3/4 above — a real `LAB-5` handler has no idea what, if anything, an earlier
unrelated event process still has in memory.

In [6]:
lab5_service_requests = fhir_search("ServiceRequest", {"identifier": SERVICE_REQUEST_PLACER_NUMBER})
assert len(lab5_service_requests) == 1
lab5_service_request = lab5_service_requests[0]
lab5_service_request_id = lab5_service_request["id"]

lab5_patient_ref = lab5_service_request["subject"]["reference"]  # e.g. "Patient/203499"
lab5_specimens = fhir_search("Specimen", {"subject": lab5_patient_ref})
assert len(lab5_specimens) == 1, "expected exactly one Specimen for this patient"
lab5_specimen_id = lab5_specimens[0]["id"]

lab5_organizations = fhir_search("Organization", {"identifier": "699X0"})
assert len(lab5_organizations) == 1, "expected the GLH Organization (ODS 699X0) to already exist"
lab5_organization_id = lab5_organizations[0]["id"]

print(f"ServiceRequest/{lab5_service_request_id}  Specimen/{lab5_specimen_id}  Organization/{lab5_organization_id} (GLH)")

ServiceRequest/205266  Specimen/206851  Organization/5 (GLH)


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised

### Build the variant `Observation`s — `05`'s pipeline, unchanged

`parse_vcf_header`/`parse_vcf_records`/the lookup tables/`build_observation()` below are
copied verbatim from `05-test-results-from-vcf.ipynb` - see that notebook for why each
mapping choice was made. `build_observation()` already takes `patient_ref`/`org_ref` as
parameters rather than hard-coding them, which is exactly what makes it reusable here
against real server-assigned ids instead of `urn:uuid` placeholders.

In [7]:
import re

VCF_PATH = Path("Input/DSS/VCF/igene_example_data.vcf")


def parse_vcf_records(path):
    records = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            if not line or line.startswith("#"):
                continue
            chrom, pos, vid, ref, alt, qual, filt, info, fmt, sample = line.split("\t")
            info_dict = {}
            for item in info.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    info_dict[k] = v
                else:
                    info_dict[item] = True
            format_dict = dict(zip(fmt.split(":"), sample.split(":")))
            records.append(
                {"chrom": chrom, "pos": int(pos), "id": vid, "ref": ref, "alt": alt, "info": info_dict, "format": format_dict}
            )
    return records


RECORDS = parse_vcf_records(VCF_PATH)

GRCH37_REFSEQ = {"17": "NC_000017.10", "15": "NC_000015.9", "X": "NC_000023.10"}
HGNC_GENE = {"BRCA1": "HGNC:1100", "FBN1": "HGNC:3603"}
SO_TERM = {
    "SNV": ("SO:0001483", "SNV"),
    "deletion": ("SO:0000159", "deletion"),
    "insertion": ("SO:0000667", "insertion"),
    "copy_number_variation": ("SO:0001019", "copy_number_variation"),
}
ALLELIC_STATE = {
    "Heterozygous": ("LA6706-1", "heterozygous"),
    "Homozygous": ("LA6705-3", "homozygous"),
    "Hemizygous": ("LA6707-9", "hemizygous"),
}
INHERITANCE_ORIGIN = {"Maternal": ("LA26320-4", "Maternal")}


def cc(system=None, code=None, display=None, text=None):
    concept = {}
    if system and code:
        coding = {"system": system, "code": code}
        if display:
            coding["display"] = display
        concept["coding"] = [coding]
    if text:
        concept["text"] = text
    elif display and "coding" not in concept:
        concept["text"] = display
    return concept


def component(loinc_code, loinc_display, **value):
    comp = {"code": {"coding": [{"system": "http://loinc.org", "code": loinc_code, "display": loinc_display}]}}
    comp.update(value)
    return comp


def dna_change_type(record):
    vartype = record["info"].get("VARTYPE", "")
    if "Copy_Number_Variant" in vartype:
        return SO_TERM["copy_number_variation"]
    if vartype == "Structural_Variant":
        return SO_TERM["deletion"] if record["info"].get("SVTYPE") == "DEL" else None
    ref, alt = record["ref"], record["alt"]
    if len(ref) == 1 and len(alt) == 1:
        return SO_TERM["SNV"]
    if len(alt) < len(ref):
        return SO_TERM["deletion"]
    if len(alt) > len(ref):
        return SO_TERM["insertion"]
    return None


def build_observation(record, obs_id, patient_ref, org_ref, effective_date):
    info = record["info"]
    fmt = record["format"]
    vartype = info.get("VARTYPE", "")
    is_structural = vartype in ("Intragenic_Copy_Number_Variant", "Multigenic_Copy_Number_Variant", "Structural_Variant")

    components = []

    gene = info.get("GENE")
    if gene:
        hgnc = HGNC_GENE.get(gene)
        components.append(component("48018-6", "Gene studied [ID]",
                                     valueCodeableConcept=cc("http://www.genenames.org", hgnc, gene) if hgnc else cc(text=gene)))

    if "INHERITANCE" in info:
        components.append(component("48002-0", "Genomic source class [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", "LA6683-2", "Germline")))

    refseq = GRCH37_REFSEQ.get(record["chrom"])
    if refseq:
        components.append(component("48013-7", "Genomic reference sequence [ID]",
                                     valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", refseq)))

    components.append(component("92822-6", "Genomic coordinate system [Type]",
                                 valueCodeableConcept=cc("http://loinc.org", "LA30102-0", "1-based character counting")))
    components.append(component("69547-8", "Genomic ref allele [ID]", valueString=record["ref"]))
    components.append(component("69551-0", "Genomic alt allele [ID]", valueString=record["alt"]))

    so_term = dna_change_type(record)
    if so_term:
        code_val, display = so_term
        components.append(component("48019-4", "DNA change type",
                                     valueCodeableConcept=cc("http://www.sequenceontology.org", code_val, display)))

    hgvsc = info.get("HGVSC")
    if hgvsc:
        transcript_match = re.match(r"(N[MR]_\d+\.\d+)", hgvsc)
        if transcript_match:
            components.append(component("51958-7", "Transcript reference sequence [ID]",
                                         valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", transcript_match.group(1))))
        components.append(component("48004-6", "DNA change (c.HGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsc)))

    hgvsp = info.get("HGVSP")
    if hgvsp:
        components.append(component("48005-3", "Amino acid change (pHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsp)))

    hgvsg = info.get("HGVSG")
    if hgvsg:
        components.append(component("81290-9", "Genomic DNA change (gHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsg)))

    cytoband = info.get("CYTOBAND")
    if cytoband:
        components.append(component("48001-2", "Cytogenetic (chromosome) location", valueCodeableConcept=cc(text=cytoband)))

    classification = info.get("CLASS")
    if classification:
        components.append(component("53037-8", "Genetic variation clinical significance [Imp]",
                                     valueCodeableConcept=cc(text=classification.replace("_", " "))))

    inheritance = info.get("INHERITANCE")
    if inheritance:
        origin = INHERITANCE_ORIGIN.get(inheritance)
        components.append(component("94186-4", "Origin of germline genetic variant [Type]",
                                     valueCodeableConcept=cc("http://loinc.org", *origin) if origin else cc(text=inheritance)))

    end = info.get("END")
    if is_structural and end:
        components.append(component("81302-2", "Structural variant inner start and end",
                                     valueRange={"low": {"value": record["pos"]}, "high": {"value": int(end)}}))
    elif not is_structural:
        components.append(component("81254-5", "Genomic allele start-end", valueRange={"low": {"value": record["pos"]}}))

    vaf = fmt.get("VAF")
    if vaf and vaf != ".":
        components.append(component("81258-6", "Sample variant allelic frequency",
                                     valueQuantity={"value": float(vaf), "unit": "decimal", "system": "http://unitsofmeasure.org"}))

    zyg = fmt.get("ZYG")
    if zyg:
        copy_match = re.search(r"\((\d+)_cop(?:y|ies)\)", zyg)
        if copy_match:
            components.append(component("82155-3", "Genomic structural variant copy number",
                                         valueQuantity={"value": int(copy_match.group(1)), "system": "http://unitsofmeasure.org", "code": "1"}))
        elif zyg in ALLELIC_STATE:
            components.append(component("53034-5", "Allelic state", valueCodeableConcept=cc("http://loinc.org", *ALLELIC_STATE[zyg])))

    return {
        "resourceType": "Observation",
        "id": obs_id,
        "meta": {"profile": [GENOMICS_VARIANT_PROFILE]},
        "status": "final",
        "category": [
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]},
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
        ],
        "code": {"coding": [{"system": "http://loinc.org", "code": "69548-6", "display": "Genetic variant assessment"}]},
        "subject": {"reference": patient_ref},
        "basedOn": [{"reference": f"ServiceRequest/{lab5_service_request_id}"}],
        "specimen": {"reference": f"Specimen/{lab5_specimen_id}"},
        "effectiveDateTime": effective_date,
        "performer": [{"reference": org_ref}],
        "valueCodeableConcept": cc("http://loinc.org", "LA9633-4", "Present"),
        "method": cc("http://loinc.org", "LA26398-0", "Sequencing"),
        "component": components,
    }


lab5_report_date = "2026-08-25T09:00:00+00:00"
lab5_observations = [
    build_observation(record, f"ctdna9737383222-{record['id'].lower()}", lab5_patient_ref, f"Organization/{lab5_organization_id}", lab5_report_date)
    for record in RECORDS
]

# A business identifier per Observation, so the transaction below can use a
# conditional (idempotent) create instead of blindly re-POSTing on every re-run.
for obs in lab5_observations:
    obs["identifier"] = [{
        "system": "https://fhir.nwgenomics.nhs.uk/Id/Notebook07VariantObservation",
        "value": obs["id"],
    }]

print(f"Built {len(lab5_observations)} variant Observation(s), referencing the real "
      f"ServiceRequest/{lab5_service_request_id} and Specimen/{lab5_specimen_id}.")

Built 4 variant Observation(s), referencing the real ServiceRequest/205266 and Specimen/206851.


Validate before sending — the same dev-loop check `05` runs, worth doing here too since
this is about to go to a real production repository, not just a local `Input/FHIR/`
file.

In [8]:
rows = []
for obs in lab5_observations:
    df = validate_resource(obs, [GENOMICS_REPORTING_IG], [GENOMICS_VARIANT_PROFILE])
    df.insert(0, "observation", obs["id"])
    rows.append(df)
pd.concat(rows, ignore_index=True)

,observation,severity,code,details,expression
0,ctdna9737383222-seqv1,information,informational,This element does not match any known slice de...,[Observation.component[11]]
1,ctdna9737383222-seqv1,information,code-invalid,Binding for path Observation.component[2].valu...,[Observation.component[2].value.ofType(Codeabl...
2,ctdna9737383222-seqv1,information,code-invalid,Binding for path Observation.component[7].valu...,[Observation.component[7].value.ofType(Codeabl...
3,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.genena...,[Observation.component[0].value.ofType(Codeabl...
4,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.genena...,[Observation.component[0].value.ofType(Codeabl...
5,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.ncbi.n...,[Observation.component[2].value.ofType(Codeabl...
6,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Observation.component[6].value.ofType(Codeabl...
7,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.sequen...,[Observation.component[6].value.ofType(Codeabl...
8,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://www.ncbi.n...,[Observation.component[7].value.ofType(Codeabl...
9,ctdna9737383222-seqv1,warning,not-found,A definition for CodeSystem 'http://varnomen.h...,[Observation.component[8].value.ofType(Codeabl...


The `information`/`warning` rows here are the same terminology-server-unavailable
limitation `03`-`06` already saw (`-tx n/a` — no network path to a real terminology
server from this notebook), not defects.

### Send as one `transaction` `Bundle`

Each entry's `request.ifNoneExist` makes this a **conditional create** — the FHIR-native
way to make a `POST` idempotent. If this notebook is re-run, the server itself skips
creating a duplicate for any `Observation` whose business identifier already exists,
rather than this notebook having to search-then-decide in Python the way section 4 did
for the `Patient` identifier.

In [9]:
transaction_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "resource": obs,
            "request": {
                "method": "POST",
                "url": "Observation",
                "ifNoneExist": f"identifier={obs['identifier'][0]['system']}|{obs['identifier'][0]['value']}",
            },
        }
        for obs in lab5_observations
    ],
}

resp = fhir_transaction(transaction_bundle)
print(f"POST {BASE_URL}  (transaction, {len(lab5_observations)} entries)  -> {resp.status_code}")

response_bundle = resp.json()
for entry in response_bundle.get("entry", []):
    status = entry.get("response", {}).get("status")
    location = entry.get("response", {}).get("location")
    print(f"  {status}  {location}")

POST https://192.168.1.62/healthconnect/cdr/fhir/r4  (transaction, 4 entries)  -> 200
  200  https://192.168.1.62/healthconnect/cdr/fhir/r4/Observation/215599
  200  https://192.168.1.62/healthconnect/cdr/fhir/r4/Observation/215600
  200  https://192.168.1.62/healthconnect/cdr/fhir/r4/Observation/215601
  200  https://192.168.1.62/healthconnect/cdr/fhir/r4/Observation/215602


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


`201 Created` with a `location` means this run actually created a new `Observation`;
`200 OK` with no new id (server-dependent - some servers still report `200`/`201` with
the existing resource's location) means the conditional create found a match from an
earlier run and did nothing further. Either way, `ServiceRequest/205266` and
`Specimen/206851` above were only ever *read*, never modified by this section - matching
the brief's own example of what a `LAB-5` does and doesn't touch.

In [10]:
verify_observations = fhir_search("Observation", {"subject": lab5_patient_ref, "code": "69548-6"})
print(f"Confirmed via a fresh search: {len(verify_observations)} variant Observation(s) now on the server for this patient.")

Confirmed via a fresh search: 4 variant Observation(s) now on the server for this patient.


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 7. `LAB-3` Laboratory Report — a status transition on `ServiceRequest`

This is the brief's headline example: "`ServiceRequest` doesn't need to be altered via
`LAB-5` or `LAB-3` other than to change status to completed." Potentially days after
`LAB-5`, the authorised report event arrives and the order can finally be marked done.

Cold lookup again — a third, independent rediscovery of the same `ServiceRequest`, on
the same principle as sections 4 and 6. This time the freshly-read `ETag` matters for a
concrete reason: by the time a real `LAB-3` arrives, something else may well have
touched this resource in between (a re-print of the order, a data-quality fix) — trusting
a version number read minutes ago in section 3 or 6 would be exactly the stale-copy bug
`If-Match` exists to prevent.

In [11]:
lab3_service_requests = fhir_search("ServiceRequest", {"identifier": SERVICE_REQUEST_PLACER_NUMBER})
assert len(lab3_service_requests) == 1
lab3_service_request_id = lab3_service_requests[0]["id"]

fresh = fhir_get(f"ServiceRequest/{lab3_service_request_id}")
lab3_etag = fresh.headers["ETag"]
current_status = fresh.json()["status"]
print(f"ServiceRequest/{lab3_service_request_id}  current status={current_status}  ETag={lab3_etag}")

ServiceRequest/205266  current status=completed  ETag=W/"12"


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### The real-world outcome here is "already completed"

`ServiceRequest/205266` is already `completed` — a prior real `LAB-3` for this order
already ran, before this notebook existed. So this section can't show a visible
`pending` → `completed` flip without pretending something happened that didn't. What it
demonstrates instead, honestly, is the two things that actually matter about this
`PATCH` in production:

1. **The correct `If-Match` succeeds** — `replace`ing `status` with the value it
   already has is a genuine no-op from a data point of view, but this server still
   accepts it as a normal write (worth knowing: some servers bump `meta.versionId` even
   when nothing semantically changed, so a `200`/`204` alone doesn't prove the *content*
   changed — only that the request was accepted).
2. **A stale/wrong `If-Match` fails safe**, `412 Precondition Failed`, rather than
   silently overwriting.

In [12]:
patch_ops = [
    {"op": "test", "path": "/status", "value": current_status},
    {"op": "replace", "path": "/status", "value": "completed"},
]

resp = fhir_patch("ServiceRequest", lab3_service_request_id, patch_ops, lab3_etag)
print(f"PATCH ServiceRequest/{lab3_service_request_id}  If-Match: {lab3_etag}  -> {resp.status_code}")

verify = fhir_get(f"ServiceRequest/{lab3_service_request_id}")
print(f"Confirmed via a fresh GET: status={verify.json()['status']}  new ETag={verify.headers['ETag']}")

PATCH ServiceRequest/205266  If-Match: W/"12"  -> 200
Confirmed via a fresh GET: status=completed  new ETag=W/"13"


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [13]:
# Same PATCH, but with a deliberately wrong If-Match - the concurrency-control path
# failing safely, not just asserted.
wrong_etag = 'W/"999"'
bad_resp = fhir_patch("ServiceRequest", lab3_service_request_id, patch_ops, wrong_etag)
print(f"PATCH ServiceRequest/{lab3_service_request_id}  If-Match: {wrong_etag} (wrong)  -> {bad_resp.status_code}")
assert bad_resp.status_code == 412, "expected a 412 Precondition Failed for a stale If-Match"
print("Confirmed: a stale/incorrect If-Match is rejected rather than silently applied.")

PATCH ServiceRequest/205266  If-Match: W/"999" (wrong)  -> 412
Confirmed: a stale/incorrect If-Match is rejected rather than silently applied.


/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 8. What an external customer is actually allowed to do

Everything above — sections 4, 6 and 7 — is internal traffic: the kind of thing the RIE
does on a real message's behalf, never exposed outside it. Per NW GM-SA's own policy
(section 1), an external GOMS-style consumer gets none of that — no `POST`, no `PUT`, no
`PATCH` — only read-only `GET`/search, the same shape of call `01`/`02` already toured
for a different resource.

The query below is what that consumer's view of this exact order looks like: the final,
authorised report, nothing about how it got there.

In [14]:
external_view = fhir_search("DiagnosticReport", {
    "subject": lab5_patient_ref,
    "status": "final",
})
for report in external_view:
    print(f"DiagnosticReport/{report['id']}  status={report['status']}  code={report['code']['coding'][0]['code']}")

print()
print("What this response does NOT reveal: which PDS check ran to verify the identifiers")
print("added in section 4, which specific ServiceRequest.status PATCH closed out the order")
print("in section 7, or that the variant Observations in section 6 arrived via a separate")
print("transaction rather than inside the report itself. That's the business logic NW GM-SA")
print("keeps internal by only exposing GET externally.")

/Users/kevinmayfield/Documents/GitHub/NHSNorthWestGMSA/Testing/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


DiagnosticReport/203502  status=final  code=R125_Cardiology
DiagnosticReport/205267  status=final  code=ctDNA_M4

What this response does NOT reveal: which PDS check ran to verify the identifiers
added in section 4, which specific ServiceRequest.status PATCH closed out the order
in section 7, or that the variant Observations in section 6 arrived via a separate
transaction rather than inside the report itself. That's the business logic NW GM-SA
keeps internal by only exposing GET externally.


## Summary

| Aggregate | Event(s) that touch it | What happened in this notebook |
|---|---|---|
| `Patient` | `LAB-1` | Located by identifier (never re-created); a demo identifier added via `PATCH`, idempotently |
| `ServiceRequest` | `LAB-4` (optional), `LAB-3` | Referenced only by `LAB-1`/`LAB-5`; `status` `PATCH`ed with `If-Match` by `LAB-3` |
| `Specimen` | `LAB-5` | Referenced only — looked up fresh, never modified |
| `Observation` (variant) | `LAB-5` | Created via a conditional-create `transaction` `Bundle` |
| `DiagnosticReport` | (pre-existing) | Read-only — the external, GOMS-style view in section 8 |

The mechanics here are the same FHIR RESTful verbs `01`/`02` already introduced
(`GET`/search, `PATCH` with `If-Match`) — this notebook's contribution is sending them
for real, against real production data, and showing that `03`-`06`'s messages and this
notebook's RESTful calls are two faces of exactly the same domain events: the RIE just
does internally, chattily, and repeatedly (once per event, cold, every time) what this
notebook did explicitly and out loud.

**What's next**, following on from `02`'s own list: category-coded searches and
pagination (`01`/`02`'s unfinished topics) apply here too; a further notebook could show
the *actual* `LAB-4` → `active` transition once a real work-order fixture exists, or
push a fresh incoming `LAB-3` conclusion (rather than an idempotent re-`PATCH` of an
already-`completed` order) against a newly-created test `ServiceRequest`.